# 4D fMRI Transformer의 Attention 메모리 간단 계산

이 노트북은 **Attention score tensor의 메모리**만 계산하고 절감 방법을 비교하였습니다. 또한 일부 방법은 계산 특성상 정확한 실측값이라고 할 수 없으며 참고용 수치임을 감안합니다.

# 챕터 1. 계산 조건 및 데이터

실제 폴더 data/localizer_first_level 데이터에서 확인한 형상만 입력값으로 사용합니다.

| 항목 | 값 | 의미 |
|---|---:|---|
| 공간 크기 | 53 × 63 × 46 | 한 frame의 voxel 수 |
| 전체 시간 길이 | 128 frames | 원본 fMRI 길이 |
| 기준 계산 길이 | 20 frames | 한 번에 처리할 시간 구간 |
| spatial patch | 6 × 6 × 6 | 한 token으로 묶을 voxel 영역 |
| temporal patch | 1 | frame마다 시간 token 생성 |
| score dtype | FP16 | 원소당 2 bytes |

공간 크기가 patch 크기 6으로 나누어떨어지지 않으므로 마지막 부분에는 패딩이 들어간다고 가정함.

In [1]:
import math
import pandas as pd

# 계산에 필요한 입력값
spatial_shape = (53, 63, 46)
frames = 20
patch_size = 6
bytes_per_value = 2  # FP16

# 챕터 2. 기준 메모리 계산

## 2-1. Token 수

$$G_x=\lceil53/6\rceil=9,\quad G_y=\lceil63/6\rceil=11,\quad G_z=\lceil46/6\rceil=8$$

20 frames를 사용할 때 전체 token 수는 다음과 같습니다.

$$N=9\times11\times8\times20=15{,}840$$

## 2-2. Attention score 메모리

Dense attention은 모든 token 쌍의 관계를 저장하므로 score 원소 수가 $N^2$입니다.

$$\text{Memory}=N^2\times\text{bytes per value}$$

FP16은 원소당 2 bytes이므로 $$15,840^2\times2 $$ 이으로 약 501.811 MB가 필요합니다.

In [2]:
# 실제 계산

# 1. 각 축의 patch 개수
spatial_grid = tuple(math.ceil(size / patch_size) for size in spatial_shape)

# 2. 전체 token 수
token_grid = spatial_grid + (frames,)
n_tokens = math.prod(token_grid)

# 3. FP16 Attention score 메모리
baseline_mb = n_tokens**2 * bytes_per_value / 1e6

pd.DataFrame({
    '항목': ['Token grid', 'Token 수 N', 'Score 원소 수 N²', 'FP16 score memory'],
    '결과': [str(token_grid), f'{n_tokens:,}', f'{n_tokens**2:,}', f'{baseline_mb:,.3f} MB']
})

,항목,결과
0,Token grid,"(9, 11, 8, 20)"
1,Token 수 N,"15,840"
2,Score 원소 수 N²,"250,905,600"
3,FP16 score memory,501.811 MB


# 챕터 3. 메모리 절감 방법

| 방법 | 원리 | 사용 목적 | 메모리 ROI가 큰 경우 | 리스크 |
|---|---|---|---|---|
| 1. Larger patch | 더 많은 voxel을 token 하나로 묶어 N을 줄임 | 입력 token 감소 | patch 확대 후 token 수가 크게 줄 때 | 세부 공간 정보 손실 |
| 2. Window attention | 작은 window 안에서만 attention 계산 | 전체 N² score 방지 | N이 window token 수보다 훨씬 클 때 | 먼 token의 직접 연결 제한 |
| 2-1. Shifted window | 다음 block에서 window 위치를 이동 | window 사이 정보 교환 | window attention을 여러 block 쓸 때 | mask와 구현 추가 |
| 3. Patch merging | 중간 단계에서 인접 spatial token을 합침 | 후반 token 감소 | N 감소가 channel 증가보다 클 때 | 공간 정보 압축 |
| 4. Temporal chunking | 긴 시간축을 여러 구간으로 나누어 처리 | peak 메모리 감소 | 전체 T가 chunk보다 클 때 | chunk 사이 attention 소실 |
| 5. FP16/BF16 | 원소 크기를 4 bytes에서 2 bytes로 줄임 | tensor 크기 절반 | 실제 16-bit로 저장될 때 | 정밀도 저하 가능 |

- 이 외에도 더 있었으나 이해 및 실제 계산 가능한 것들만 작성함

# 챕터 4. 방법별 메모리 계산

## 4-1. Larger patch와 Window attention

- **Larger patch:** patch를 6에서 12로 키운 뒤 token 수를 다시 계산함.
- **Window attention:** window 수 × window 내부 token 수의 제곱으로 계산함.

$$\text{Window elements}=\text{window 수}\times(\text{window 내부 token 수})^2$$

In [3]:
def dense_score_mb(patch, time, value_bytes=2):
    grid = tuple(math.ceil(size / patch) for size in spatial_shape) + (time,)
    tokens = math.prod(grid)
    return tokens**2 * value_bytes / 1e6

def window_score_mb(grid, window, value_bytes=2):
    window_count = math.prod(math.ceil(g / w) for g, w in zip(grid, window))
    tokens_per_window = math.prod(window)
    return window_count * tokens_per_window**2 * value_bytes / 1e6

patch12_mb = dense_score_mb(12, 20)
window4_mb = window_score_mb(token_grid, (4, 4, 4, 4))
window2_mb = window_score_mb(token_grid, (2, 2, 2, 2))

pd.DataFrame([
    ['Dense baseline', baseline_mb, 1.0],
    ['Larger patch 12³', patch12_mb, baseline_mb / patch12_mb],
    ['Window 4⁴', window4_mb, baseline_mb / window4_mb],
    ['Window 2⁴', window2_mb, baseline_mb / window2_mb],
    ['Shifted window 4⁴', window4_mb, baseline_mb / window4_mb],
], columns=['방법', 'Score memory (MB)', '절감 배수']).style.format({
    'Score memory (MB)': '{:,.3f}', '절감 배수': '{:,.2f}배'
}).hide(axis='index')

방법,Score memory (MB),절감 배수
Dense baseline,501.811,1.00배
Larger patch 12³,11.520,43.56배
Window 4⁴,11.796,42.54배
Window 2⁴,0.614,816.75배
Shifted window 4⁴,11.796,42.54배


## 4-2. Patch merging

한 번의 merging마다 spatial grid의 각 축을 절반으로 줄이고 시간축은 유지함.

첫 번째 merging에서는 (9, 11, 8, 20)이 (5, 6, 4, 20)으로 바뀜.

In [4]:
merge_grid = token_grid
merge_results = []

for stage in range(1, 4):
    spatial_part = tuple(math.ceil(g / 2) for g in merge_grid[:3])
    merge_grid = spatial_part + (merge_grid[3],)
    merge_tokens = math.prod(merge_grid)
    merge_mb = merge_tokens**2 * bytes_per_value / 1e6
    merge_results.append([stage, merge_grid, merge_tokens, merge_mb, baseline_mb / merge_mb])

pd.DataFrame(
    merge_results,
    columns=['Merging 횟수', 'Token grid', 'Token 수', 'Score memory (MB)', '절감 배수']
).style.format({
    'Score memory (MB)': '{:,.4f}', '절감 배수': '{:,.2f}배'
}).hide(axis='index')

Merging 횟수,Token grid,Token 수,Score memory (MB),절감 배수
1,"(5, 6, 4, 20)",2400,11.5200,43.56배
2,"(3, 3, 2, 20)",360,0.2592,"1,936.00배"
3,"(2, 2, 1, 20)",80,0.0128,"39,204.00배"


## 4-3. Temporal chunking과 FP16

- **Temporal chunking:** 128 frames 전체와 20 frames씩 처리할 때의 peak score를 비교함.
- **FP16/BF16:** 같은 tensor를 FP32 4 bytes와 FP16/BF16 2 bytes로 저장할 때를 비교함.

In [9]:
# 128 frames 전체와 20-frame chunk 비교
full128_mb = dense_score_mb(6, 128)

# 같은 tensor의 FP32와 FP16 비교
fp32_mb = n_tokens**2 * 4 / 1e6
fp16_mb = n_tokens**2 * 2 / 1e6

pd.DataFrame([
    ['Temporal: 128 → 20 frames', full128_mb, baseline_mb, full128_mb / baseline_mb],
    ['Precision: FP32 → FP16/BF16', fp32_mb, fp16_mb, fp32_mb / fp16_mb],
], columns=['비교', '변경 전 (MB)', '변경 후 (MB)', '절감 배수']).style.format({
    '변경 전 (MB)': '{:,.3f}', '변경 후 (MB)': '{:,.3f}', '절감 배수': '{:,.2f}배'
}).hide(axis='index')

비교,변경 전 (MB),변경 후 (MB),절감 배수
Temporal: 128 → 20 frames,"20,554.187",501.811,40.96배
Precision: FP32 → FP16/BF16,"1,003.622",501.811,2.00배


# 챕터 5. 결과 정리

| 방법 | 계산된 score 메모리 | 기준 대비 |
|---|---:|---:|
| Dense FP16 기준 | 501.811 MB | 1배 |
| Larger patch 12³ | 11.520 MB | 43.56배 감소 |
| Window 4⁴ | 11.796 MB | 42.54배 감소 |
| Window 2⁴ | 0.614 MB | 816.75배 감소 |
| Shifted window 4⁴ | 11.796 MB | 42.54배 감소 |
| Patch merging 1회 | 11.520 MB | 43.56배 감소 |
| Patch merging 2회 | 0.259 MB | 1,936배 감소 |
| Patch merging 3회 | 0.0128 MB | 39,204배 감소 |
| 128 → 20-frame chunk | peak 501.811 MB | 40.96배 감소 |
| FP32 → FP16/BF16 | 1,003.622 → 501.811 MB | 2배 감소 |

## 결론

이론적인 score 메모리만 보면 **작은 window**와 **여러 단계의 patch merging**이 가장 큰 감소를 보임. 다만 메모리 절감량만 계산했으며 정확도, 정보 손실과 실행 시간은 측정하지 않았으므로 어느 방법이 최종적으로 가장 우수하다고 단정하기 어렵습니다.

하나의 구조적 방법을 선택한다면 감소량을 예측하기 쉽고 목적이 분명한 **Window attention**이 적합합니다. 현재 데이터 규모에서는 여러 단계의 patch merging이 과도한 공간 정보 압축을 일으킬 가능성이 있습니다.